In [4]:
PROJECT_ID  = "qwiklabs-gcp-02-e659e41ff1eb"          # ⟵ change
LOCATION    = "us"                      # BigQuery / Vertex AI region
DATASET_ID  = "weather_ds"              # new or existing dataset
TABLE_ID    = "weather_raw"
MODEL_ID    = "gemini_weather_model"
ENRICHED_TABLE_ID = "weather_with_report"

import os, google.cloud.bigquery as bq
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
client = bq.Client(project=PROJECT_ID, location=LOCATION)
print("BigQuery client initialised.")

BigQuery client initialised.


In [5]:
dataset_ref = bq.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
dataset_ref.location = LOCATION
client.delete_dataset(dataset_ref, delete_contents=True, not_found_ok=True)  # idempotent
client.create_dataset(dataset_ref)
print(f"Dataset `{DATASET_ID}` ready.")

Dataset `weather_ds` ready.


In [6]:
gcs_uri = "gs://labs.roitraining.com/data-to-ai-workshop/weather_data.csv"

job_cfg = bq.LoadJobConfig(
    source_format=bq.SourceFormat.CSV,
    skip_leading_rows=1,
    autodetect=True,
)
load_job = client.load_table_from_uri(
    gcs_uri,
    f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}",
    job_config=job_cfg,
)
load_job.result()
print(f"Loaded {load_job.output_rows:,} rows into `{TABLE_ID}`.")

Loaded 300 rows into `weather_raw`.


In [7]:
preview_df = client.query(f"""
SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
LIMIT 5
""").to_dataframe()
preview_df

,date,city,state,temperature_f,wind_speed_mph,precipitation_in,barometric_pressure_inHg,humidity_percent,weather_condition
0,2025-02-21,Atlanta,GA,55.7,5.0,0.12,29.80,50.4,Cloudy
1,2025-02-26,Atlanta,GA,75.2,10.4,0.03,29.58,49.9,Cloudy
2,2025-03-01,Atlanta,GA,51.7,4.7,0.08,29.74,49.9,Cloudy
3,2025-03-05,Atlanta,GA,74.4,5.1,0.02,29.92,50.4,Cloudy
4,2025-03-10,Atlanta,GA,59.5,9.6,0.09,29.67,57.2,Cloudy


In [8]:
create_model_sql = f"""
CREATE OR REPLACE MODEL `{PROJECT_ID}.{DATASET_ID}.{MODEL_ID}` REMOTE
WITH CONNECTION DEFAULT
OPTIONS(
  ENDPOINT  = 'gemini-2.5-flash'
);
"""
client.query(create_model_sql).result()
print("Gemini-Flash registered in BigQuery ML ✓")

Gemini-Flash registered in BigQuery ML ✓


In [9]:
gen_sql = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.{ENRICHED_TABLE_ID}` AS
SELECT
  w.*,
  ml_generate_text_result AS weather_report
FROM
  `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}` AS w
JOIN
  ML.GENERATE_TEXT(
      MODEL `{PROJECT_ID}.{DATASET_ID}.{MODEL_ID}`,
      (
        SELECT
          CONCAT(
            'Generate a short weather report or warning. ',
            'Location City=', CAST(w.city AS STRING), ', ',
            'Temp=', CAST(w.temperature_f AS STRING), '°C, ',
            'Humidity=', CAST(w.humidity_percent AS STRING), '%, ',
            'Wind=', CAST(w.wind_speed_mph AS STRING), ' km/h, ',
            'Pressure=', CAST(w.barometric_pressure_inHg AS STRING), ' mb.'
          ) AS prompt
        FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}` AS w
      ),
      STRUCT(
        0.2  AS temperature,        -- creativity
        256  AS max_output_tokens
      )
  ) AS result
ON TRUE   -- CROSS JOIN each generated row back to its prompt row
;
"""
client.query(gen_sql).result()
print(f"Created table `{ENRICHED_TABLE_ID}` with weather reports ✓")

Created table `weather_with_report` with weather reports ✓


In [2]:
result=  ML.GENERATE_TEXT(
      MODEL `{PROJECT_ID}.{DATASET_ID}.{MODEL_ID}`,
      (
        SELECT
          CONCAT(
            'Generate a short weather report or warning. ',
            'Location City=', CAST(w.city AS STRING), ', ',
            'Temp=', CAST(w.temperature_f AS STRING), '°C, ',
            'Humidity=', CAST(w.humidity_percent AS STRING), '%, ',
            'Wind=', CAST(w.wind_speed_mph AS STRING), ' km/h, ',
            'Pressure=', CAST(w.barometric_pressure_inHg AS STRING), ' mb.'
          ) AS prompt
        FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}` AS w
      ),
      STRUCT(
        0.2  AS temperature,        -- creativity
        256  AS max_output_tokens
      )
  )

SyntaxError: invalid syntax (121438988.py, line 2)